In [1]:
import nflreadpy as nfl
import pandas as pd

In [2]:
SEASON = 2025
 
pbp = nfl.load_pbp(seasons=[SEASON]).to_pandas()
 

In [4]:
#Load and filter to real 4th-down decisions

fourth = pbp[
    (pbp["down"] == 4)
    & (pbp["play_type"].isin(["run", "pass", "punt", "field_goal"]))
    & (pbp["qb_kneel"].fillna(0) != 1)
    & (pbp["qb_spike"].fillna(0) != 1)
].copy()

fourth["decision"] = fourth["play_type"].map(
    {"run": "go", "pass": "go", "punt": "punt", "field_goal": "field_goal"}
)

In [ ]:
# Quarter labels (nflverse uses qtr = 5 for overtime)
quarter_order = ["Q1", "Q2", "Q3", "Q4", "OT"]
fourth["quarter"] = pd.Categorical(
    fourth["qtr"].map({1: "Q1", 2: "Q2", 3: "Q3", 4: "Q4", 5: "OT"}),
    categories=quarter_order,
    ordered=True,
)

In [6]:
# Yards-to-go buckets (right-inclusive: 0-3, 4-7, 8-10, 11+)
fourth["ytg_bucket"] = pd.cut(
    fourth["ydstogo"],
    bins=[0, 3, 7, 10, float("inf")],
    labels=["0-3", "3-7", "7-10", "10+"],
    include_lowest=True,
)

In [ ]:
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
 
# Overall average yards to go by decision (for reference)
overall = (
    fourth.groupby("decision")["ydstogo"]
    .agg(plays="count", avg_yards_to_go="mean")
    .round(2)
)
print(f"\n=== {SEASON}: overall average yards to go by decision ===")
print(overall)


=== 2025: overall average yards to go by decision ===
            plays  avg_yards_to_go
decision                          
field_goal   1024             8.03
go            931             3.63
punt         2042             9.26


In [ ]:
# Average yards to go by quarter and decision
avg_by_qtr = fourth.pivot_table(
    index="quarter", columns="decision", values="ydstogo",
    aggfunc="mean", observed=False,
).round(2)
print(f"\n=== {SEASON}: average yards to go by quarter and decision ===")
print(avg_by_qtr)


=== 2025: average yards to go by quarter and decision ===
decision  field_goal    go   punt
quarter                          
Q1              7.91  1.99   8.54
Q2              7.92  2.22   9.64
Q3              7.93  2.76   9.21
Q4              8.38  5.14   9.53
OT              7.43  3.56  11.12


In [ ]:
# Decision counts and mix by quarter
counts_by_qtr = pd.crosstab(
    fourth["quarter"], fourth["decision"], margins=True, margins_name="total"
)
print(f"\n=== {SEASON}: number of decisions by quarter ===")
print(counts_by_qtr)
 
mix_by_qtr = (
    pd.crosstab(fourth["quarter"], fourth["decision"], normalize="index") * 100
).round(1)
print(f"\n=== {SEASON}: % of decisions within each quarter ===")
print(mix_by_qtr)


=== 2025: number of decisions by quarter ===
decision  field_goal   go  punt  total
quarter                               
Q1               208  135   478    821
Q2               349  192   601   1142
Q3               205  172   462    839
Q4               255  423   493   1171
OT                 7    9     8     24
total           1024  931  2042   3997

=== 2025: % of decisions within each quarter ===
decision  field_goal    go  punt
quarter                         
Q1              25.3  16.4  58.2
Q2              30.6  16.8  52.6
Q3              24.4  20.5  55.1
Q4              21.8  36.1  42.1
OT              29.2  37.5  33.3


In [ ]:
# Yards-to-go bucket breakdown within each quarter 
counts_qtr_bucket = pd.crosstab(
    [fourth["quarter"], fourth["ytg_bucket"]], fourth["decision"]
)
print(f"\n=== {SEASON}: decisions by quarter and yards-to-go bucket ===")
print(counts_qtr_bucket)
 
mix_qtr_bucket = (
    pd.crosstab(
        [fourth["quarter"], fourth["ytg_bucket"]],
        fourth["decision"],
        normalize="index",
    ) * 100
).round(1)
print(f"\n=== {SEASON}: % of decisions within each quarter/bucket ===")
print(mix_qtr_bucket)


=== 2025: decisions by quarter and yards-to-go bucket ===
decision            field_goal   go  punt
quarter ytg_bucket                       
Q1      0-3                 27  115    95
        3-7                 93   19   137
        7-10                43    1   106
        10+                 45    0   140
Q2      0-3                 44  161    83
        3-7                155   24   177
        7-10                71    4   134
        10+                 79    3   207
Q3      0-3                 32  128    90
        3-7                 75   33   120
        7-10                51    7    96
        10+                 47    4   156
Q4      0-3                 41  208    70
        3-7                 80  109   150
        7-10                62   56    96
        10+                 72   50   177
OT      0-3                  0    5     2
        3-7                  5    3     0
        7-10                 0    1     2
        10+                  2    0     4

=== 2025: % of d

In [ ]:
# Go-for-it rate: quarter x bucket grid
fourth["is_go"] = (fourth["decision"] == "go").astype(int)
 
go_rate = (
    fourth.pivot_table(
        index="quarter", columns="ytg_bucket", values="is_go",
        aggfunc="mean", observed=False,
    ) * 100
).round(1)
print(f"\n=== {SEASON}: go-for-it rate (%) by quarter and yards-to-go bucket ===")
print(go_rate)
 
sample_size = fourth.pivot_table(
    index="quarter", columns="ytg_bucket", values="is_go",
    aggfunc="count", observed=False,
)
print(f"\n=== {SEASON}: sample size behind each go-rate cell ===")
print(sample_size)


=== 2025: go-for-it rate (%) by quarter and yards-to-go bucket ===
ytg_bucket   0-3   3-7  7-10   10+
quarter                           
Q1          48.5   7.6   0.7   0.0
Q2          55.9   6.7   1.9   1.0
Q3          51.2  14.5   4.5   1.9
Q4          65.2  32.2  26.2  16.7
OT          71.4  37.5  33.3   0.0

=== 2025: sample size behind each go-rate cell ===
ytg_bucket  0-3  3-7  7-10  10+
quarter                        
Q1          237  249   150  185
Q2          288  356   209  289
Q3          250  228   154  207
Q4          319  339   214  299
OT            7    8     3    6
